In [1]:
import numpy as np
import gc
from matplotlib import pyplot as plt
import time 

import sys
sys.path.append('/dipc/kstoreyf/muchisimocks/scripts')
import compute_statistics as cs
import data_loader
import utils
import plotter

import bacco

%load_ext autoreload
%autoreload 2

2026-03-10 19:57:33.568846: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-10 19:57:33.670825: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-10 19:57:33.670869: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-10 19:57:33.683541: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-10 19:57:33.714131: I tensorflow/core/platform/cpu_feature_guar

# Load tracer field

In [2]:
tag_params = '_shame_p0_n1000'
box_size = 1000.0
dir_mocks = f'/scratch/kstoreyf/muchisimocks/muchisimocks_lib{tag_params}'
idx_mock = 0
fn_fields = f'{dir_mocks}/mock{idx_mock}/bias_fields_eul_deconvolved_{idx_mock}.npy'
bias_terms_eul = np.load(fn_fields)

In [3]:
params_df, param_dict_fixed = data_loader.load_cosmo_params(tag_params)
if params_df is None:
    param_dict = param_dict_fixed
else:
    param_dict = params_df.iloc[0].to_dict()
    param_dict.update(param_dict_fixed)
cosmo = utils.get_cosmo(param_dict)

In [4]:
tag_mock = '_nbar0.00022'
param_dict_forbias = data_loader.load_params_ood('shame', tag_mock)
bias_vector = [param_dict_forbias[name] for name in utils.biasparam_names_ordered]

In [5]:
n_grid_orig = 512
bs = np.concatenate(([1.0], bias_vector))
tracer_field_det = utils.get_tracer_field(bias_terms_eul, bias_vector, n_grid_orig, 
                                             noise_field=None, A_noise=None)

# Power spectra

In [6]:
k_min = 0.01
k_max = 0.1
n_bins = 3

In [7]:
n_grid = tracer_field_det.shape[-1]
args_power_grid = {
    "ngrid": n_grid,
    "box": box_size,
    "interlacing": False, #default: True
    "deposit_method": 'cic', #default: "tsc",
    "log_binning": True,
    "kmin": k_min,
    "kmax": k_max,
    "nbins": n_bins,
    "cosmology": cosmo,
}

In [8]:
pk_obj = bacco.statistics.compute_crossspectrum_twogrids(
                    grid1=tracer_field_det,
                    grid2=tracer_field_det,
                    **args_power_grid)

2026-03-10 19:57:43,267 bacco.statistics : Computing the power spectrum with ngrid=128 and interlacing=False


bacco.power : boxsize 1000.000000; ngrid 128; nthreads 64; interlacing 0; deposit_method 1; log_binning 1; type 1; precision=single; correct_grid=1 (log=1); correct_sn=0
bacco.power : normalise_grid1=0 normalise_grid2=0 deconvolve_grid1=0 deconvolve_grid2=0


2026-03-10 19:57:45,449 bacco.util : pk multipoles at k 0.17782795284786496 set to zero: it seems you have a lot of bins for this grid size
2026-03-10 19:57:45,450 bacco.statistics :  ...done in 2.18 s


bacco.power : total mass 0.999999 (grid1) 0.999999 (grid2)
bacco.power : Doing FFTW
bacco.power: FFT took 0.099241 sec
bacco.power : Counting modes
bacco.power : done counting modes in 0.199243 sec
bacco.power : Starting Fourier loop 
bacco.power : Doing inverse FFT
bacco.power : Creating final array
bacco.power : done Fourier loop in 0.100870 secs
bacco.power : Deallocating arrays


In [9]:
print('pk_obj["k"]:', pk_obj['k'])
print('pk_obj["pk"]:', pk_obj['pk'])
print('pk_obj["nmodes"]:', pk_obj['nmodes'])

pk_obj["k"]: [0.01778279 0.05623413 0.17782795]
pk_obj["pk"]: [62608.52045965 26891.01413977     0.        ]
pk_obj["nmodes"]: [  496 16364     1]


In [10]:
r_edges = np.logspace(np.log10(k_min), np.log10(k_max), n_bins+1)
r_centers = np.sqrt(r_edges[:-1] * r_edges[1:])
print('Expected r_edges:', r_edges)
print('Expected r_centers:', r_centers)

print("-- match what bacco code is actually doing --")
dlog = np.log10(k_max/k_min)/(n_bins-1)
r_edges_match = np.logspace(np.log10(k_min), np.log10(k_max) + dlog, n_bins+1)
print('Matched r_edges:', r_edges_match)
r_centers_match = np.sqrt(r_edges_match[:-1] * r_edges_match[1:])
print('Matched r_centers:', r_centers_match)
print(" -- show what bacco code gives --")
print('pk_obj["k"]:', pk_obj['k'])
print(" -- way to get bacco to do what we want: nbins->nbins+1, cut off last bin --")
r_edges_corr = np.logspace(np.log10(k_min), np.log10(k_max)+dlog, n_bins+2)
print('Corrected r_edges:', r_edges_corr)
r_centers_corr = np.sqrt(r_edges_corr[:-1] * r_edges_corr[1:])
print('Corrected r_centers:', r_centers_corr)

Expected r_edges: [0.01       0.02154435 0.04641589 0.1       ]
Expected r_centers: [0.01467799 0.03162278 0.06812921]
-- match what bacco code is actually doing --
Matched r_edges: [0.01       0.03162278 0.1        0.31622777]
Matched r_centers: [0.01778279 0.05623413 0.17782794]
 -- show what bacco code gives --
pk_obj["k"]: [0.01778279 0.05623413 0.17782795]
 -- way to get bacco to do what we want: nbins->nbins+1, cut off last bin --
Corrected r_edges: [0.01       0.02371374 0.05623413 0.13335214 0.31622777]
Corrected r_centers: [0.01539927 0.03651741 0.08659643 0.2053525 ]


In [11]:
k_min = 0.01
k_max = 0.1
n_bins = 3+1

n_grid = tracer_field_det.shape[-1]
args_power_grid = {
    "ngrid": n_grid,
    "box": box_size,
    "interlacing": False, #default: True
    "deposit_method": 'cic', #default: "tsc",
    "log_binning": True,
    "kmin": k_min,
    "kmax": k_max,
    "nbins": n_bins,
    "cosmology": cosmo,
}

pk_obj = bacco.statistics.compute_crossspectrum_twogrids(
                    grid1=tracer_field_det,
                    grid2=tracer_field_det,
                    **args_power_grid)
print('pk_obj["k"]:', pk_obj['k'])
print('pk_obj["pk"]:', pk_obj['pk'])
print('pk_obj["nmodes"]:', pk_obj['nmodes'])

2026-03-10 19:57:50,447 bacco.statistics : Computing the power spectrum with ngrid=128 and interlacing=False
2026-03-10 19:57:50,949 bacco.util : pk multipoles at k 0.14677993526075359 set to zero: it seems you have a lot of bins for this grid size
2026-03-10 19:57:50,953 bacco.statistics :  ...done in 0.506 s


bacco.power : boxsize 1000.000000; ngrid 128; nthreads 64; interlacing 0; deposit_method 1; log_binning 1; type 1; precision=single; correct_grid=1 (log=1); correct_sn=0
bacco.power : normalise_grid1=0 normalise_grid2=0 deconvolve_grid1=0 deconvolve_grid2=0
bacco.power : total mass 0.999999 (grid1) 0.999999 (grid2)
bacco.power : Doing FFTW
bacco.power: FFT took 0.091542 sec
bacco.power : Counting modes
bacco.power : done counting modes in 0.200000 sec
bacco.power : Starting Fourier loop 
bacco.power : Doing inverse FFT
bacco.power : Creating final array
bacco.power : done Fourier loop in 0.099779 secs
bacco.power : Deallocating arrays
pk_obj["k"]: [0.01467799 0.03162278 0.06812921 0.14677994]
pk_obj["pk"]: [62930.4992947  43617.13476724 23569.2030488      0.        ]
pk_obj["nmodes"]: [  152  1572 15136     1]


In [12]:
# clearly show what bacco code is actually doing
k_min, k_max, n_bins = 0.1, 1.0, 3

K0 = k_min
K1 = k_max
bins_ps = n_bins

binfac = (bins_ps - 1) / np.log(K1 / K0)
print(binfac)

edges = [K0 * np.exp(i / binfac) for i in range(bins_ps + 1)]
centers = [np.exp((i + 0.5) / binfac + np.log(K0)) for i in range(bins_ps)]

print("edges:  ", edges)
print("centers:", centers)

0.8685889638065037
edges:   [0.1, 0.31622776601683794, 0.9999999999999999, 3.162277660168378]
centers: [0.1778279410038923, 0.5623413251903491, 1.7782794100389225]


In [13]:
# what bacco code should be doing
k_min, k_max, n_bins = 0.1, 1.0, 3

K0 = k_min
K1 = k_max
bins_ps = n_bins

# This is the part that changes
binfac_corr = (bins_ps) / np.log(K1 / K0)
print(binfac)

edges = [K0 * np.exp(i / binfac_corr) for i in range(bins_ps + 1)]
centers = [np.exp((i + 0.5) / binfac_corr + np.log(K0)) for i in range(bins_ps)]

print("edges:  ", edges)
print("centers:", centers)

0.8685889638065037
edges:   [0.1, 0.21544346900318834, 0.46415888336127786, 0.9999999999999999]
centers: [0.14677992676220697, 0.31622776601683794, 0.6812920690579614]


In [14]:
# how to get bacco to do what we want
k_min = 0.1
k_max = 1.0
n_bins = 3

dlog = np.log10(k_max/k_min)/(n_bins-1)
print(dlog)

k_max_pass = 10 ** (np.log10(k_max) - np.log10(k_max/k_min)/n_bins)
print(k_max_pass)
dlog_match = np.log10(k_max_pass/k_min)/(n_bins-1)
print(np.exp(np.log10(k_max_pass) + dlog_match))


print((dlog_match))
r_edges_match = np.logspace(np.log10(k_min), np.log10(k_max_pass) + dlog_match, n_bins+1)
print('Matched r_edges:', r_edges_match)
r_centers_match = np.sqrt(r_edges_match[:-1] * r_edges_match[1:])
print('Matched r_centers:', r_centers_match)

n_grid = tracer_field_det.shape[-1]
args_power_grid_corr = {
    "ngrid": n_grid,
    "box": box_size,
    "interlacing": False, #default: True
    "deposit_method": 'cic', #default: "tsc",
    "log_binning": True,
    "kmin": k_min,
    "kmax": k_max_pass,
    "nbins": n_bins,
    "cosmology": cosmo,
}

pk_obj_pass = bacco.statistics.compute_crossspectrum_twogrids(
                    grid1=tracer_field_det,
                    grid2=tracer_field_det,
                    **args_power_grid_corr)

print(pk_obj_pass['k'])
print(pk_obj_pass['pk'])

2026-03-10 19:57:52,178 bacco.statistics : Computing the power spectrum with ngrid=128 and interlacing=False


0.5
0.4641588833612779
1.0
0.33333333333333337
Matched r_edges: [0.1        0.21544347 0.46415888 1.        ]
Matched r_centers: [0.14677993 0.31622777 0.68129207]
bacco.power : boxsize 1000.000000; ngrid 128; nthreads 64; interlacing 0; deposit_method 1; log_binning 1; type 1; precision=single; correct_grid=1 (log=1); correct_sn=0
bacco.power : normalise_grid1=0 normalise_grid2=0 deconvolve_grid1=0 deconvolve_grid2=0
bacco.power : total mass 0.999999 (grid1) 0.999999 (grid2)
bacco.power : Doing FFTW
bacco.power: FFT took 0.061394 sec
bacco.power : Counting modes


2026-03-10 19:57:52,549 bacco.util : pk multipoles at k 0.6812919869591569 set to zero: it seems you have a lot of bins for this grid size
2026-03-10 19:57:52,550 bacco.statistics :  ...done in 0.372 s


bacco.power : done counting modes in 0.101183 sec
bacco.power : Starting Fourier loop 
bacco.power : Doing inverse FFT
bacco.power : Creating final array
bacco.power : done Fourier loop in 0.100128 secs
bacco.power : Deallocating arrays
[0.14677992 0.31622774 0.68129199]
[10330.56330301  5156.0614132      0.        ]


In [ ]:
pk_obj_pass

{'k': array([0.14677992, 0.31622777, 0.68129211, 1.4677994 ]),
 'pk': array([10330.56330301,  5156.0614132 ,  1645.20979954,     0.        ]),
 'pk1': array([10330.56330301,  5156.0614132 ,  1645.20979954,     0.        ]),
 'pk2': array([10330.56330301,  5156.0614132 ,  1645.20979954,     0.        ]),
 'pk_l2': array([-56.50798228, -43.23733495, -25.60483076,   0.        ]),
 'pk_l4': array([  101.24401707, -1868.06211119, -3815.46873142,     0.        ]),
 'pk_theory_lin': array([3293.05883789,  809.43096924,  161.94255066,    0.        ]),
 'pk_theory_nl': array([3750.47950349, 1460.07966356,  420.63066696,    0.        ]),
 'shotnoise': array([1.00000072e+09, 1.00000072e+09, 1.00000072e+09, 1.00000072e+09]),
 'nmodes': array([ 152054, 1389817,  538402,       1]),
 'pk_gaussian_error': array([37.46623704,  6.18520744,  3.17090297,  0.        ]),
 'r': array([  3.328125,   7.984375,  12.640625,  17.296875,  21.953125,
         26.609375,  31.265625,  35.921875,  40.578125,  45.23437

In [19]:
pk_obj_corr = pk_obj_pass.copy()
for key in pk_obj_pass:
    if len(pk_obj_pass[key])==len(pk_obj_pass['k']):
        pk_obj_corr[key] = pk_obj_pass[key][:-1]
print(pk_obj_corr)

{'k': array([0.14677992, 0.31622777, 0.68129211]), 'pk': array([10330.56330301,  5156.0614132 ,  1645.20979954]), 'pk1': array([10330.56330301,  5156.0614132 ,  1645.20979954]), 'pk2': array([10330.56330301,  5156.0614132 ,  1645.20979954]), 'pk_l2': array([-56.50798228, -43.23733495, -25.60483076]), 'pk_l4': array([  101.24401707, -1868.06211119, -3815.46873142]), 'pk_theory_lin': array([3293.05883789,  809.43096924,  161.94255066]), 'pk_theory_nl': array([3750.47950349, 1460.07966356,  420.63066696]), 'shotnoise': array([1.00000072e+09, 1.00000072e+09, 1.00000072e+09]), 'nmodes': array([ 152054, 1389817,  538402]), 'pk_gaussian_error': array([37.46623704,  6.18520744,  3.17090297]), 'r': array([  3.328125,   7.984375,  12.640625,  17.296875,  21.953125,
        26.609375,  31.265625,  35.921875,  40.578125,  45.234375,
        49.890625,  54.546875,  59.203125,  63.859375,  68.515625,
        73.171875,  77.828125,  82.484375,  87.140625,  91.796875,
        96.453125, 101.109375, 10

In [ ]:
# how to get bacco to do what we want
k_min = 0.1
k_max = 1.0
n_bins = 3

print((dlog_match))
r_edges_target = np.logspace(np.log10(k_min), np.log10(k_max), n_bins+1)
print('Target r_edges:', r_edges_target)
r_centers_target = np.sqrt(r_edges_target[:-1] * r_edges_target[1:])
print('Target r_centers:', r_centers_target)

n_grid = tracer_field_det.shape[-1]
args_power_grid_corr = {
    "ngrid": n_grid,
    "box": box_size,
    "interlacing": False, #default: True
    "deposit_method": 'cic', #default: "tsc",
    "log_binning": True,
    "kmin": k_min,
    "kmax": k_max,
    "nbins": n_bins+1,
    "cosmology": cosmo,
}

pk_obj_pass = bacco.statistics.compute_crossspectrum_twogrids(
                    grid1=tracer_field_det,
                    grid2=tracer_field_det,
                    **args_power_grid_corr)

print(pk_obj_pass['k'])
print(pk_obj_pass['pk'])

2026-03-10 20:00:25,273 bacco.statistics : Computing the power spectrum with ngrid=128 and interlacing=False


0.25
Target r_edges: [0.1        0.21544347 0.46415888 1.        ]
Target r_centers: [0.14677993 0.31622777 0.68129207]
bacco.power : boxsize 1000.000000; ngrid 128; nthreads 64; interlacing 0; deposit_method 1; log_binning 1; type 1; precision=single; correct_grid=1 (log=1); correct_sn=0
bacco.power : normalise_grid1=0 normalise_grid2=0 deconvolve_grid1=0 deconvolve_grid2=0
bacco.power : total mass 0.999999 (grid1) 0.999999 (grid2)
bacco.power : Doing FFTW
bacco.power: FFT took 0.151769 sec
bacco.power : Counting modes
bacco.power : done counting modes in 0.103127 sec
bacco.power : Starting Fourier loop 
bacco.power : Doing inverse FFT


2026-03-10 20:00:25,852 bacco.util : pk multipoles at k 1.4677993995410619 set to zero: it seems you have a lot of bins for this grid size
2026-03-10 20:00:25,853 bacco.statistics :  ...done in 0.58 s


bacco.power : Creating final array
bacco.power : done Fourier loop in 0.100193 secs
bacco.power : Deallocating arrays
[0.14677992 0.31622777 0.68129211 1.4677994 ]
[10330.56330301  5156.0614132   1645.20979954     0.        ]


In [24]:
pnn = cs.compute_pnn_from_bias_fields(bias_terms_eul, cosmo, box_size, n_grid_orig,
                        k_min=0.01, k_max=0.4, n_bins=32)

Computing the 15 PNN cross power spectra
Computing cross-spectrum 1 of 15


2026-03-10 21:08:05,821 bacco.statistics : Computing the power spectrum with ngrid=128 and interlacing=False
2026-03-10 21:08:05,903 bacco.statistics :  ...done in 0.0819 s


bacco.power : boxsize 1000.000000; ngrid 128; nthreads 1; interlacing 0; deposit_method 1; log_binning 1; type 1; precision=single; correct_grid=0 (log=1); correct_sn=0
bacco.power : normalise_grid1=0 normalise_grid2=0 deconvolve_grid1=0 deconvolve_grid2=0
bacco.power : total mass 1 (grid1) 1 (grid2)
bacco.power : Doing FFTW
bacco.power: FFT took 0.014137 sec
bacco.power : Counting modes
bacco.power : done counting modes in 0.042720 sec
bacco.power : Starting Fourier loop 
bacco.power : done Fourier loop in 0.000144 secs
bacco.power : Deallocating arrays
Computing cross-spectrum 2 of 15


2026-03-10 21:08:05,908 bacco.statistics : Computing the power spectrum with ngrid=128 and interlacing=False
2026-03-10 21:08:06,005 bacco.statistics :  ...done in 0.0966 s


bacco.power : boxsize 1000.000000; ngrid 128; nthreads 1; interlacing 0; deposit_method 1; log_binning 1; type 1; precision=single; correct_grid=0 (log=1); correct_sn=0
bacco.power : normalise_grid1=0 normalise_grid2=0 deconvolve_grid1=0 deconvolve_grid2=0
bacco.power : total mass 1 (grid1) 4.48822e-11 (grid2)
bacco.power : Doing FFTW
bacco.power: FFT took 0.014405 sec
bacco.power : Counting modes
bacco.power : done counting modes in 0.043591 sec
bacco.power : Starting Fourier loop 
bacco.power : done Fourier loop in 0.000139 secs
bacco.power : Deallocating arrays
Computing cross-spectrum 3 of 15


...

bacco.power : done counting modes in 0.041988 sec
bacco.power : Starting Fourier loop 
bacco.power : done Fourier loop in 0.000137 secs
bacco.power : Deallocating arrays
Computing cross-spectrum 15 of 15


2026-03-10 21:08:07,145 bacco.statistics : Computing the power spectrum with ngrid=128 and interlacing=False


bacco.power : boxsize 1000.000000; ngrid 128; nthreads 1; interlacing 0; deposit_method 1; log_binning 1; type 1; precision=single; correct_grid=0 (log=1); correct_sn=0
bacco.power : normalise_grid1=0 normalise_grid2=0 deconvolve_grid1=0 deconvolve_grid2=0
bacco.power : total mass -3.83591e-13 (grid1) -3.83591e-13 (grid2)
bacco.power : Doing FFTW
bacco.power: FFT took 0.013897 sec
bacco.power : Counting modes


2026-03-10 21:08:07,240 bacco.statistics :  ...done in 0.0954 s


bacco.power : done counting modes in 0.043226 sec
bacco.power : Starting Fourier loop 
bacco.power : done Fourier loop in 0.000140 secs
bacco.power : Deallocating arrays


In [27]:
r_edges = np.logspace(np.log10(0.01), np.log10(0.4), 32+1)
r_centers = np.sqrt(r_edges[:-1] * r_edges[1:])
print("Expected r_edges:", r_edges)
print("Expected r_centers:", r_centers)


Expected r_edges: [0.01       0.01122185 0.01259299 0.01413166 0.01585833 0.01779598
 0.01997038 0.02241045 0.02514867 0.02822145 0.03166969 0.03553924
 0.03988159 0.04475452 0.05022284 0.0563593  0.06324555 0.0709732
 0.07964504 0.08937645 0.1002969  0.11255165 0.12630375 0.14173615
 0.15905415 0.17848814 0.20029668 0.22476988 0.25223334 0.28305242
 0.31763711 0.35644754 0.4       ]
Expected r_centers: [0.01059332 0.01188767 0.01334016 0.01497012 0.01679924 0.01885185
 0.02115526 0.02374011 0.0266408  0.0298959  0.03354872 0.03764786
 0.04224786 0.0474099  0.05320267 0.05970323 0.06699805 0.0751842
 0.08437056 0.09467936 0.10624774 0.11922959 0.13379763 0.15014567
 0.16849118 0.18907824 0.21218073 0.23810598 0.26719891 0.29984655
 0.33648323 0.37759636]


In [ ]:
pnn

[{'k': array([0.01059332, 0.01188766, 0.01334016, 0.01497012, 0.01679924,
         0.01885185, 0.02115526, 0.02374011, 0.02664079, 0.02989589,
         0.03354872, 0.03764786, 0.04224786, 0.0474099 , 0.05320267,
         0.05970323, 0.06699805, 0.07518419, 0.08437056, 0.09467936,
         0.10624773, 0.11922959, 0.13379763, 0.15014567, 0.16849118,
         0.18907824, 0.21218073, 0.23810598, 0.26719891, 0.29984656,
         0.33648324, 0.37759637]),
  'pk': array([27872.87967586, 36717.73688436, 20990.45672982, 21441.75243274,
         34015.69038883, 29786.1067212 , 42288.52699812, 26486.48199759,
         21784.83713085, 22865.37499603, 19229.15653811, 16374.80496865,
         15071.67384594, 11721.29423649, 11381.66565212, 10985.32815569,
         10383.98758023,  8913.31305989,  7555.39784771,  6025.42032758,
          5032.97889944,  4620.12340387,  3999.45340908,  3384.95230238,
          2856.12894527,  2488.73581322,  2101.17799672,  1821.74225298,
          1557.64456534,  135

## Linear

In [13]:
n_grid = tracer_field_det.shape[-1]
args_power_grid_lin = {
    "ngrid": n_grid,
    "box": box_size,
    "interlacing": False, #default: True
    "deposit_method": 'cic', #default: "tsc",
    "log_binning": False,
    "kmin": k_min,
    "kmax": k_max,
    "nbins": n_bins,
    "cosmology": cosmo,
}
pk_obj_lin = bacco.statistics.compute_crossspectrum_twogrids(
                    grid1=tracer_field_det,
                    grid2=tracer_field_det,
                    **args_power_grid_lin)

2026-02-26 17:10:18,267 bacco.statistics : Computing the power spectrum with ngrid=128 and interlacing=False
2026-02-26 17:10:18,727 bacco.util : pk multipoles at k 0.8500000002483528 set to zero: it seems you have a lot of bins for this grid size
2026-02-26 17:10:18,728 bacco.statistics :  ...done in 0.46 s


bacco.power : boxsize 1000.000000; ngrid 128; nthreads 64; interlacing 0; deposit_method 1; log_binning 0; type 1; precision=single; correct_grid=1 (log=1); correct_sn=0
bacco.power : normalise_grid1=0 normalise_grid2=0 deconvolve_grid1=0 deconvolve_grid2=0
bacco.power : total mass 0.999999 (grid1) 0.999999 (grid2)
bacco.power : Doing FFTW
bacco.power: FFT took 0.047079 sec
bacco.power : Counting modes
bacco.power : done counting modes in 0.101387 sec
bacco.power : Starting Fourier loop 
bacco.power : Doing inverse FFT
bacco.power : Creating final array
bacco.power : done Fourier loop in 0.101916 secs
bacco.power : Deallocating arrays


In [14]:
r_edges_lin = np.linspace(k_min, k_max, n_bins+1)
r_centers_lin = (r_edges_lin[:-1] + r_edges_lin[1:]) / 2
print("Expected r_edges_lin:", r_edges_lin)
print("Expected r_centers_lin:", r_centers_lin)

print("pk_obj_lin['k']:", pk_obj_lin['k'])

Expected r_edges_lin: [0.1 0.4 0.7 1. ]
Expected r_centers_lin: [0.25 0.55 0.85]
pk_obj_lin['k']: [0.25 0.55 0.85]
